# Building RAG Applications on Databricks with Vector Search, FM APIs, and LangChain
In this series we look at how you can build managed RAG applications on Databricks utilizing Databricks Vector Search. In coming notebooks we'll expand to evaluating RAG systems and introducing the idea of LLM as a Judge.

### Additional Resources/Credits
- Intro Page: https://www.databricks.com/product/machine-learning/vector-search
- Vector Search Docs: https://docs.databricks.com/aws/en/vector-search/vs-example-notebooks
- LangChain Databricks: https://docs.langchain.com/oss/python/integrations/vectorstores/databricks_vector_search

## Setup
Ensure that you first execute the ```upload_data.py``` script to create a catalog, schema, and volume with your PDF data. Feel free to replace the data with your own documents, if you have a custom RAG use-case that you are testing.

In [0]:
%pip install -U databricks-vectorsearch databricks-langchain langchain langchain-text-splitters pypdf

In [0]:
dbutils.library.restartPython()

## Read Data from Volume & Prepare
Here we ensure we can access the data in Unity Catalog and prepare it for applying our Embeddings model.

In [0]:
os.listdir("/Volumes/demo_rag_catalog/demo_rag_data/pdf_source_files/")

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader

VOLUME_PATH = "/Volumes/demo_rag_catalog/demo_rag_data/pdf_source_files/"

all_docs = []

for f in os.listdir(VOLUME_PATH):
    if f.lower().endswith(".pdf"):
        print(f)
        path = os.path.join(VOLUME_PATH, f)
        loader = PyPDFLoader(path)
        all_docs.extend(loader.load())

print("Total pages:", len(all_docs))

### Chunk Text

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = splitter.split_documents(all_docs)

print("Total chunks:", len(chunks))
print(chunks[0].page_content[:300])

In [0]:
for d in chunks:
    print(d.page_content)
    print("----------")
    print(d.metadata.get("source", "unknown"))
    print(d.metadata.get("page", None))

## Upload Chunked Data to Unity Catalog

In [0]:
import uuid
import pandas as pd

rows = []
for d in chunks:
    rows.append({
        "chunk_id": str(uuid.uuid4()),
        "source": d.metadata.get("source", "unknown"),
        "page": d.metadata.get("page", None),
        "content": d.page_content
    })

chunk_df = spark.createDataFrame(pd.DataFrame(rows))

# adjust for your catalog and schema name
CATALOG = "demo_rag_catalog"
SCHEMA  = "demo_rag_data"

CHUNKS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.rag_chunks"
print(CHUNKS_TABLE)

# create schema if needed
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")

# write table
chunk_df.write.mode("overwrite").saveAsTable(CHUNKS_TABLE)

# view
display(spark.table(CHUNKS_TABLE).limit(5))

## Create Embeddings via Vector Search
We enable a Vector Search Endpoint & Index, this will be our retrieval engine, which we later encapsulate with LangChain constructs.

In [0]:
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.rag_chunks"
# Index table in unity catalog
VS_INDEX     = f"{CATALOG}.{SCHEMA}.ragchunksindex"

# VS ENDPOINT
VS_ENDPOINT = "rag-vs-endpoint-medium"

# Embedding Model ID, check what you have available in Serving section
EMBEDDING_ENDPOINT = "databricks-bge-large-en"

### Vector Search EP & Index Creation
This step can take a little depending on amount of data you have, this example should be ~10 minutes.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if VS_ENDPOINT not in existing:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
else:
    print("Endpoint already exists")

In [0]:
# Enable Change Data Feed on the source table
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Check if index already exists
existing_indexes = vsc.list_indexes(VS_ENDPOINT).get("vector_indexes", [])
existing_names = [i["name"] for i in existing_indexes]

if VS_INDEX not in existing_names:
    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=CHUNKS_TABLE,
        primary_key="chunk_id",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
        pipeline_type="TRIGGERED"
    )
else:
    print("Index already exists")

In [0]:
import time

TERMINAL_STATES = {
    "ONLINE_NO_PENDING_UPDATE",
    "ONLINE_WITH_PENDING_UPDATE"
}

while True:
    desc = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
    status = desc.get("status", {})
    
    detailed_state = status.get("detailed_state")
    print("Index state:", detailed_state)

    if detailed_state in TERMINAL_STATES:
        print("Index is ready.")
        break

    time.sleep(60)

## Query Retriever

In [0]:
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

resp = index.similarity_search(
    query_text="What are SageMaker Multi-Model Endpoints?",
    columns=["chunk_id", "source", "page", "content"],
    num_results=5
)
resp

## LangChain Integration
We now wrap these constructs with LangChain-Databricks integration package. We provide our models via the FM APIs.

### Enable Retriever

In [0]:
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
vector_store = DatabricksVectorSearch(
    index_name=VS_INDEX,
)

In [0]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
retriever.invoke("What are SageMaker Multi-Model Endpoints?")

### Build LLM Construct & Chain

In [0]:
# Specify FM Model ID from Serving
llm = ChatDatabricks(
    endpoint="databricks-qwen3-next-80b-a3b-instruct",
    temperature=0.1,
    max_tokens=500
)

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the question, for AWS and SageMaker related questions, please use the context provided to answer the question:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [0]:
import mlflow
# Enable tracing via MLflow
mlflow.langchain.autolog()

response = rag_chain.invoke("What is the difference between SageMaker Multi-Model and Multi-Container Endpoints?")
print(response)